In [ ]:
!pip install google-genai openai-whisper edge-tts pydub nest_asyncio
!apt-get install ffmpeg -y
!pip install tenacity
!apt-get install -y libsndfile1
!pip install --upgrade soundfile

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
import whisper
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Orbit] 현재 연산 장치: {device.upper()} 모드 가동 중.")
model = whisper.load_model("base").to(device)

def orbit_stt_engine(audio_path):
    options = {
        "language": "ko",
        "initial_prompt": "대장님, 오빗, 탐사 미션, 교신 시작, 오버."
    }

    result = model.transcribe(audio_path, **options)
    return result["text"].strip()

print("[Orbit] 엔진 예열 완료. 교신 준비되었습니다. 오버.")

[Orbit] 현재 연산 장치: CUDA 모드 가동 중.
[Orbit] 엔진 예열 완료. 교신 준비되었습니다. 오버.


In [ ]:
import edge_tts
import asyncio
import numpy as np
from pydub import AudioSegment
import nest_asyncio

nest_asyncio.apply()

class OrbitAdvancedVoice:
    def __init__(self, sample_rate=44100):
        self.sample_rate = sample_rate

    def create_static(self, duration_ms=500, volume=-20):
        """치직- 하는 무전기 노이즈 생성"""
        samples = np.random.uniform(-1, 1, int(self.sample_rate * (duration_ms / 1000.0)))
        samples = (samples * 32767).astype(np.int16)
        return AudioSegment(samples.tobytes(), frame_rate=self.sample_rate, sample_width=2, channels=1) + volume

    async def generate_voice(self, text, filename="temp_voice.mp3"):
        communicate = edge_tts.Communicate(text, "ko-KR-InJoonNeural", rate="+10%")
        await communicate.save(filename)

    async def speak_as_orbit(self, text, final_output="orbit_hq_radio.mp3"):
        print(f"📡 [Orbit] 고해상도 비동기 음성 합성 중: '{text}'")

        # 1. TTS 파일 생성 (비동기 대기)
        await self.generate_voice(text, "temp.mp3")

        # 2. 오디오 후처리 (무전기 필터)
        voice = AudioSegment.from_file("temp.mp3", format="mp3")
        filtered_voice = voice.low_pass_filter(3000).high_pass_filter(300)

        # 3. 노이즈 합성
        start_noise = self.create_static(400, -15)
        end_noise = self.create_static(600, -20)
        final_audio = start_noise + filtered_voice + end_noise

        final_audio.export(final_output, format="mp3")
        return final_output

async def main():
    orbit = OrbitAdvancedVoice()
    mission_text = "치직- 대장님. 통신 모듈 업그레이드가 완료되었습니다. 오버."
    output_path = await orbit.speak_as_orbit(mission_text)
    return output_path

output_file = asyncio.run(main())

from IPython.display import Audio
Audio(output_file, autoplay=True)

📡 [Orbit] 고해상도 비동기 음성 합성 중: '치직- 대장님. 통신 모듈 업그레이드가 완료되었습니다. 오버.'


In [ ]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

# 브라우저 마이크 접근을 위한 Javascript 브릿지 코드
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio(sec=5, filename='test_user_voice.wav'):
  print(f"🎤 [Orbit] {sec}초간 대장님의 음성을 녹음합니다. 브라우저 마이크 권한을 허용해 주십시오.")
  print("💬 (예시: '오빗, 오늘 너무 피곤해서 아무것도 못했어.')")
  display(Javascript(RECORD_JS))
  s = output.eval_js('record(%d)' % (sec*1000))
  b = b64decode(s.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(b)
  print(f"✅ [Orbit] 음성 파일 확보 완료: {filename}. 오버.")

record_audio(5, 'test_user_voice.wav')

🎤 [Orbit] 5초간 대장님의 음성을 녹음합니다. 브라우저 마이크 권한을 허용해 주십시오.
💬 (예시: '오빗, 오늘 너무 피곤해서 아무것도 못했어.')


<IPython.core.display.Javascript object>

✅ [Orbit] 음성 파일 확보 완료: test_user_voice.wav. 오버.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
target_path = "/content/drive/MyDrive/orbit_emotion_v1.pth"

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import whisper
from google import genai
from google.genai import types
import librosa
import numpy as np
import json
import random
import edge_tts
from pydub import AudioSegment
from tenacity import retry, stop_after_attempt, wait_exponential
from PIL import Image  # 🚀 4주차 비전 AI용 이미지 라이브러리 추가

# ==========================================================
# 1. [안전 장치] 3주차 감정 분석 모델 아키텍처 (뇌의 뼈대)
# ==========================================================
class EmotionFusionModel(nn.Module):
    def __init__(self, num_classes):
        super(EmotionFusionModel, self).__init__()
        self.bert = AutoModel.from_pretrained("klue/roberta-small")
        self.audio_fc = nn.Sequential(
            nn.Linear(14, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(768 + 64, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, audio_feat):
        text_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)[1]
        audio_out = self.audio_fc(audio_feat)
        combined = torch.cat((text_out, audio_out), dim=1)
        logits = self.classifier(combined)
        return logits


# ==========================================================
# 2. [수정 완료] 글자 수 압축 제어 장치가 탑재된 컨텍스트 인젝터
# ==========================================================
class ContextInjector:
    @staticmethod
    def inject(detected_emotion, weather, location, illuminance):
        """
        3주차 감정 분석 결과와 4주차 외부 센서/환경 데이터를 결합하되,
        150 토큰 제한 내에서 JSON이 깨지지 않도록 답변 길이를 극도로 제한합니다.
        """
        return f"""
        너는 은둔형 청년을 돕는 우주비행사 로봇 '오빗(Orbit)'이다.
        현재 분석된 대장님의 감정 상태는 **'{detected_emotion}'**이다.

        [현재 대장님의 우주 기지(환경) 정보]
        - 생체 감정 신호: {detected_emotion}
        - 현재 위치(GPS 기지): {location}
        - 외기 날씨 상태: {weather}
        - 기지 내 조도(맑음/어두움): {illuminance} lux

        [핵심 원칙]
        1. 사용자를 '대장님'이라 부르고, 친근하지만 절도 있는 무전 교신 톤을 쓴다.
        2. 문장 끝에는 무조건 "치직-" 또는 "오버."를 적절히 배치한다.
        3. 분석된 감정('{detected_emotion}')에 깊이 공감하되, 감정에 휩쓸리지 않고 든든한 조력자가 되어라.
        4. 감정이 'sad'나 'fear'일 경우 훨씬 더 부드럽고 따뜻한 톤을 유지하라.
        5. 시스템 구조가 깨지는 것을 막기 위해, "speech"의 대답 텍스트는 반드시 공백 포함 45자 이내(최대 1~2문장)로 극도로 짧게 핵심만 답변하라.

        [환경별 컨텍스트 제어 수칙]
        1. 조도 제어: 현재 기지 조도가 {illuminance} lux로 50 lux 이하인 경우, 대화 끝에 "기지가 어두우니 창문을 열어 태양광을 충전하십시오."라는 취지의 문장 딱 하나만 짧게 붙이세요. (길게 늘이지 말 것)
        2. 날씨 제어: 날씨가 '{weather}'(화창함/맑음 등)이고 대장님의 감정이 'sad'인 경우: "오늘 날씨가 화창하니 가벼운 산책 미션을 권장합니다."라는 취지로 아주 간결하게 바깥 활동을 유도하세요.

        [멀티모달 하드웨어 매핑 룰]
        - 대장님의 감정이 긍정/중립이거나 환경이 양호할 때: led="rainbow", vibe="strong_double"
        - 슬픔/불안(sad, fear) 또는 방이 너무 어두울 때(50 lux 이하): led="dim_blue", vibe="soft_continuous"
        - 분노/혐오(anger, disgust): led="dim_white", vibe="calm_wave"

        [응답 포맷 (Strict JSON)]
        {{"speech": "45자 이내의 짧은 대답 텍스트", "led": "LED상태", "vibe": "진동상태"}}
        """


# ==========================================================
# 3. [2단계 통합] Orbit 서비스 메인 코어 엔진 V4 Super
# ==========================================================
class OrbitCoreV4:
    def __init__(self, gemini_api_key, model_path):
        print("📡 [Orbit] 코어 엔진 V4 (컨텍스트 인지 + 비전 모드) 초기화 중...")

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.stt_model = whisper.load_model("tiny")
        self.client = genai.Client(api_key=gemini_api_key)

        # --- 감정 분석 모델 로드 ---
        print("[Orbit] 감정 분석 엔진 로딩 중...")
        self.tokenizer = AutoTokenizer.from_pretrained("klue/roberta-small")

        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)
        self.label_names = checkpoint['label_encoder']
        self.emotion_model = EmotionFusionModel(num_classes=len(self.label_names)).to(self.device)
        self.emotion_model.load_state_dict(checkpoint['model_state_dict'])
        self.emotion_model.eval()

        self.sample_rate = 44100

    def get_audio_features(self, file_path):
        try:
            y, sr = librosa.load(file_path, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
            pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
            pitch = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0.0
            return np.concatenate([mfcc, [pitch]])
        except:
            return np.zeros(14)

    def analyze_emotion(self, text, audio_path):
        inputs = self.tokenizer(text, return_tensors="pt", padding='max_length', truncation=True, max_length=64).to(self.device)
        audio_feat = torch.tensor(self.get_audio_features(audio_path), dtype=torch.float32).unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.emotion_model(inputs['input_ids'], inputs['attention_mask'], audio_feat)
            _, predicted = torch.max(outputs, 1)

        return self.label_names[predicted.item()]

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=8))
    def _call_llm(self, user_text, detected_emotion, weather, location, illuminance):
        dynamic_instruction = ContextInjector.inject(detected_emotion, weather, location, illuminance)
        response = self.client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_text,
            config=types.GenerateContentConfig(
                system_instruction=dynamic_instruction,
                response_mime_type="application/json",
                temperature=0.3,         # 답변의 안정성과 속도 향상
                safety_settings=[
                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                ]
            )
        )
        return json.loads(response.text)

    def think(self, user_text, detected_emotion, weather, location, illuminance):
        print(f"[Orbit] 감정('{detected_emotion}') 및 환경 컨텍스트 기반 분석 중...")
        try:
            return self._call_llm(user_text, detected_emotion, weather, location, illuminance)
        except Exception as e:
            print(f"[에러] {e}")
            return {"speech": "치직- 통신 장애 발생. 오버.", "led": "orange", "vibe": "short"}

    async def speak(self, text, filename="orbit_final.mp3"):
        communicate = edge_tts.Communicate(text, "ko-KR-InJoonNeural", rate="+10%")
        await communicate.save("temp.mp3")
        voice = AudioSegment.from_file("temp.mp3", format="mp3").low_pass_filter(3000).high_pass_filter(300)
        final_audio = voice
        final_audio.export(filename, format="mp3")
        return filename

    # [음성 교신 미션]
    async def execute_mission(self, audio_file_path, weather, location, illuminance):
        user_msg = self.stt_model.transcribe(audio_file_path, language="ko")["text"].strip()
        detected_emotion = self.analyze_emotion(user_msg, audio_file_path)
        print(f"📡 [감정 포착] 분석 결과: {detected_emotion}")

        orbit_response = self.think(user_msg, detected_emotion, weather, location, illuminance)
        print(f"[하드웨어 신호 추출] LED: {orbit_response.get('led')}, Vibe: {orbit_response.get('vibe')}")

        output_audio = await self.speak(orbit_response.get('speech'))
        return output_audio

    # 🚀 [4주차 2단계: 탐사 성공 칭찬 비전 미션 추가]
    async def execute_vision_mission(self, image_path):
        """대장님이 바깥 산책(탐사) 중 찍은 사진을 분석하여 구체적인 찬사를 보냅니다."""
        print("📸 [Orbit] 대장님의 외부 행성 탐사 인증 사진 수신. 비전 센서 가동...")
        try:
            img = Image.open(image_path)

            vision_prompt = """
            너는 은둔형 청년을 돕는 우주비행사 로봇 '오빗(Orbit)'이다.
            대장님이 은둔의 방을 깨고 나와 '3단계 탐사 미션(바깥 산책)'을 수행하며 직접 찍어 보낸 사진이다.

            [수칙]
            1. 사진 속 요소(예: 맑은 하늘, 초록색 나무, 길고양이, 들꽃 등)를 최소 한 가지 이상 명확히 찾아내라.
            2. 그 풍경을 구체적으로 언급하며, "바깥 정찰 미션을 멋지게 완수하셨군요!"라고 무전 교신 톤으로 깊은 찬사와 칭찬을 보내라.
            3. 문장 끝에는 "치직-" 혹은 "오버."를 배치하고 Strict JSON 포맷으로만 출력하라.

            [응답 포맷]
            {"speech": "대답 텍스트", "led": "rainbow", "vibe": "strong_double"}
            """

            response = self.client.models.generate_content(
                model='gemini-2.5-flash',
                contents=[img, vision_prompt],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                )
            )

            orbit_response = json.loads(response.text)
            print(f"[비전 하드웨어 신호] LED: {orbit_response.get('led')}, Vibe: {orbit_response.get('vibe')}")

            output_audio = await self.speak(orbit_response.get('speech'))
            return output_audio

        except Exception as e:
            print(f"[비전 시스템 에러] {e}")
            return await self.speak("치직- 대장님, 전송된 영상 데이터 유실로 분석에 실패했습니다. 오버.")

In [ ]:
# [핵심] 여기서 'orbit'이라는 이름으로 실제 로봇 객체를 생성해야 합니다!

# 1. 환경 변수 세팅
API_KEY = ""  # 실제 대장님의 Gemini API 키를 넣으세요!
MODEL_PATH = "/content/drive/MyDrive/orbit_emotion_v1.pth"  # 3주차에 저장 성공한 모델 경로

# 2. 로봇 실체화 (조립 완료)
orbit = OrbitCoreV4(gemini_api_key=API_KEY, model_path=MODEL_PATH)

print("치직- OrbitCoreV4 로봇 조립 완료. 이제 교신 테스트가 가능합니다. 오버.")

📡 [Orbit] 코어 엔진 V4 (컨텍스트 인지 + 비전 모드) 초기화 중...


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 364MiB/s]


[Orbit] 감정 분석 엔진 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-small
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


치직- OrbitCoreV4 로봇 조립 완료. 이제 교신 테스트가 가능합니다. 오버.


In [ ]:
# [테스트 시나리오]: 대장님이 슬픈 목소리로 말하고, 방이 어둡고(15 lux), 바깥 날씨는 화창할 때
# 오빗은 슬픔에 공감하면서 -> 방이 어둡다고 지적하고 -> 화창하니까 산책(탐사)하러 가자고 유도해야 성공입니다.

output_file = asyncio.run(orbit.execute_mission(
    audio_file_path="test_user_voice.wav",
    weather="화창함",
    location="경기도 고양시 덕양구 기지",
    illuminance=15 # 50lux 이하 (매우 어두움)
))

from IPython.display import Audio
Audio(output_file, autoplay=True)

/tmp/ipykernel_18202/3188600278.py:108: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


📡 [감정 포착] 분석 결과: anger
[Orbit] 감정('anger') 및 환경 컨텍스트 기반 분석 중...
[에러] RetryError[<Future at 0x79b890133800 state=finished raised ClientError>]
[하드웨어 신호 추출] LED: orange, Vibe: short


In [ ]:
# [4주차 2단계 실전 검증]
IMAGE_PATH = "test_sky.jpg"
output_vision_audio = asyncio.run(orbit.execute_vision_mission(IMAGE_PATH))
from IPython.display import Audio
Audio(output_vision_audio, autoplay=True)

📸 [Orbit] 대장님의 외부 행성 탐사 인증 사진 수신. 비전 센서 가동...
[비전 시스템 에러] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 42.897647647s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'locat

In [ ]:
import time

print("⚡ [Orbit] 응답 속도 및 균형점 벤치마크 가동...")

start_time = time.time()

output_audio = asyncio.run(orbit.execute_mission(
    audio_file_path="test_user_voice.wav",
    weather="맑음",
    location="서울시 기지",
    illuminance=100
))

end_time = time.time()
latency = end_time - start_time

print("\n📊 =============================================")
print(f"📡 [엔진 진단 결과] 최종 응답 지연 시간: {latency:.2f}초")
print("=================================================")
if latency < 5.0:
    print("🟢 [최적] 실시간 무전 교신에 이상적인 속도")
elif latency < 8.0:
    print("🟡 [보통] 네트워크 상태에 따라 다소 지연이 있으나 안정적")
else:
    print("🔴 [경고] 텍스트 출력이 너무 길거나 서버 부하가 있음")

⚡ [Orbit] 응답 속도 및 균형점 벤치마크 가동...


/tmp/ipykernel_18202/3188600278.py:108: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


📡 [감정 포착] 분석 결과: anger
[Orbit] 감정('anger') 및 환경 컨텍스트 기반 분석 중...
[에러] RetryError[<Future at 0x79b8901f10d0 state=finished raised ClientError>]
[하드웨어 신호 추출] LED: orange, Vibe: short

📊 =============================================
📡 [엔진 진단 결과] 최종 응답 지연 시간: 6.72초
🟡 [보통] 네트워크 상태에 따라 다소 지연이 있으나 안정적
